In [1]:

from pathlib import Path
import json
import gc
import os

import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from transformers import Qwen2AudioForConditionalGeneration
from numcodecs import Quantize, Zstd


# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "submission_artifacts"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Export both variants
EXPORT_VARIANTS = [
    {
        "name": "selected_layer_codec_q2_zstd",
        "codec_digits": 2,
        "codec_zstd_level": 3,
        "layer_subset": "all_selected",
    },
    {
        "name": "selected_layer_codec_q3_zstd",
        "codec_digits": 3,
        "codec_zstd_level": 3,
        "layer_subset": "all_selected",
    },
]


# ============================================================
# HELPERS
# ============================================================

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class CodecPipeline:
    def __init__(self, filters, compressor=None, encoded_dtype="f2"):
        self.filters = filters
        self.compressor = compressor
        self.encoded_dtype = np.dtype(encoded_dtype)

    def encode(self, arr):
        x = np.asarray(arr, dtype=np.float32)
        for f in self.filters:
            x = f.encode(x)
        x = np.asarray(x, dtype=self.encoded_dtype)
        filtered_shape = x.shape
        filtered_bytes = x.tobytes(order="C")
        if self.compressor is not None:
            filtered_bytes = self.compressor.encode(filtered_bytes)
        return {
            "payload": filtered_bytes,
            "filtered_shape": filtered_shape,
            "filtered_dtype": self.encoded_dtype.str,
        }

    def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
        payload = encoded_obj["payload"]
        filtered_shape = tuple(encoded_obj["filtered_shape"])
        filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])
        if self.compressor is not None:
            payload = self.compressor.decode(payload)
        x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)
        for f in reversed(self.filters):
            x = f.decode(x)
        return np.asarray(x, dtype=original_dtype).reshape(original_shape)


class CompressedLinear(nn.Module):
    def __init__(self, linear_layer, pipeline, description):
        super().__init__()
        weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)
        self.weight_shape = weight.shape
        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.pipeline = pipeline
        self.description = description
        self.encoded_weight = self.pipeline.encode(weight)

        if linear_layer.bias is not None:
            self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
        else:
            self.bias = None

        self._cached_weight = None
        self._cached_device = None
        self._cached_dtype = None

    def _get_cached_weight(self, device, dtype):
        if self._cached_weight is None or self._cached_device != device or self._cached_dtype != dtype:
            decoded = self.pipeline.decode(self.encoded_weight, self.weight_shape, np.float32)
            self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
            self._cached_device = device
            self._cached_dtype = dtype
        return self._cached_weight

    def forward(self, x):
        weight = self._get_cached_weight(x.device, x.dtype)
        return torch.nn.functional.linear(x, weight, self.bias)


def get_module_by_name(model, module_name):
    module = model
    for part in module_name.split("."):
        module = getattr(module, part)
    return module


def set_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def get_linear_layer_names(model):
    return [name for name, module in model.named_modules() if isinstance(module, nn.Linear)]


def should_select_layer(layer_name: str) -> bool:
    return any(pattern in layer_name for pattern in ["mlp", "feed_forward", "up_proj", "down_proj", "gate_proj"])


def should_keep_for_subset(layer_name: str, subset: str) -> bool:
    if subset == "all_selected":
        return should_select_layer(layer_name)
    if subset == "up_down_only":
        return ("up_proj" in layer_name) or ("down_proj" in layer_name)
    if subset == "down_only":
        return "down_proj" in layer_name
    if subset == "up_only":
        return "up_proj" in layer_name
    if subset == "gate_only":
        return "gate_proj" in layer_name
    raise ValueError(f"Unsupported subset: {subset}")


def get_selected_linear_layer_names(model, subset="all_selected"):
    return [name for name in get_linear_layer_names(model) if should_keep_for_subset(name, subset)]


def estimate_saved_dir_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total


def save_codec_artifact(model, save_dir: Path, metadata: dict):
    save_dir.mkdir(parents=True, exist_ok=True)

    # Save only metadata/config sidecars here. The actual compressed model is stored
    # as a Torch object because HuggingFace save_pretrained does not know CompressedLinear.
    torch_path = save_dir / "compressed_model.pt"
    meta_path = save_dir / "compression_metadata.json"

    torch.save(model, torch_path)

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return estimate_saved_dir_bytes(save_dir)


def export_variant(cfg):
    print(f"\n=== Exporting {cfg['name']} ===")

    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map=None,
    )
    model = model.to(DEVICE)
    model.eval()

    selected_layers = get_selected_linear_layer_names(model, subset=cfg["layer_subset"])
    pipeline = CodecPipeline(
        filters=[Quantize(digits=cfg["codec_digits"], dtype="f4", astype="f2")],
        compressor=Zstd(level=cfg["codec_zstd_level"]),
        encoded_dtype="f2",
    )
    desc = (
        f"Selected-layer codec: subset={cfg['layer_subset']}, "
        f"Quantize(digits={cfg['codec_digits']}, astype='f2') + Zstd(level={cfg['codec_zstd_level']})"
    )

    applied = []
    for layer_name in tqdm(selected_layers, desc=f"compress_{cfg['name']}"):
        layer = get_module_by_name(model, layer_name)
        if not isinstance(layer, nn.Linear):
            continue
        compressed_layer = CompressedLinear(layer, pipeline, desc)
        compressed_layer = compressed_layer.to(next(model.parameters()).device)
        set_module_by_name(model, layer_name, compressed_layer)
        applied.append(layer_name)

    save_dir = OUTPUT_ROOT / cfg["name"]
    metadata = {
        "base_model_id": MODEL_ID,
        "method": "selected_layer_codec",
        "setting": f"q{cfg['codec_digits']} + zstd-{cfg['codec_zstd_level']}",
        "layer_subset": cfg["layer_subset"],
        "selected_layer_count": len(selected_layers),
        "applied_layer_count": len(applied),
        "selected_layers": selected_layers,
        "applied_layers": applied,
        "description": desc,
        "device_used_for_export": DEVICE,
    }
    artifact_bytes = save_codec_artifact(model, save_dir, metadata)
    metadata["artifact_bytes"] = int(artifact_bytes)
    metadata["artifact_gb"] = artifact_bytes / (1024 ** 3)

    with open(save_dir / "compression_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"saved: {save_dir}")
    print(f"artifact size: {metadata['artifact_gb']:.3f} GB")

    del model
    cleanup()


def main():
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))

    for cfg in EXPORT_VARIANTS:
        export_variant(cfg)

    manifest = {
        "base_model_id": MODEL_ID,
        "artifacts_root": str(OUTPUT_ROOT),
        "variants": [v["name"] for v in EXPORT_VARIANTS],
    }
    with open(OUTPUT_ROOT / "submission_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print("\nDone.")
    print("Artifacts root:", OUTPUT_ROOT)


if __name__ == "__main__":
    main()


cuda available: True
gpu: NVIDIA RTX A6000

=== Exporting selected_layer_codec_q2_zstd ===


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

compress_selected_layer_codec_q2_zstd:   0%|          | 0/96 [00:00<?, ?it/s]

saved: /home/paperspace/projects/iwslt2026-compression/outputs/submission_artifacts/selected_layer_codec_q2_zstd
artifact size: 10.687 GB

=== Exporting selected_layer_codec_q3_zstd ===


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

compress_selected_layer_codec_q3_zstd:   0%|          | 0/96 [00:00<?, ?it/s]

saved: /home/paperspace/projects/iwslt2026-compression/outputs/submission_artifacts/selected_layer_codec_q3_zstd
artifact size: 13.530 GB

Done.
Artifacts root: /home/paperspace/projects/iwslt2026-compression/outputs/submission_artifacts
